# Predict One Match With Version 2

This notebook uses the existing Version 2 architecture with Version 2 features:

- CatBoostClassifier for outcome probabilities
- leakage-safe recent form features from `data/raw/results.csv`
- latest Elo data plus FIFA validation and optional snapshot features
- CatBoostRegressor goal models plus statistical xG ensemble
- calibrated Poisson score matrix
- default highest score-probability selector

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
if (cwd / "core_prediction" / "scripts").exists():
    PROJECT_ROOT = cwd
elif cwd.name == "core_prediction":
    PROJECT_ROOT = cwd.parent
elif cwd.name == "notebooks" and cwd.parent.name == "core_prediction":
    PROJECT_ROOT = cwd.parents[1]
else:
    PROJECT_ROOT = cwd

sys.path.append(str(PROJECT_ROOT / "core_prediction" / "scripts"))

from predict_single_match_v2 import (
    RESULTS_PATH,
    load_fixtures,
    load_elo,
    load_fifa,
    load_results,
    validate_fixture,
    build_feature_row,
    load_prediction_models,
    load_goal_ensemble_config,
    load_score_selection_config,
    predict_outcome_probabilities,
    predict_ensemble_expected_goals,
    generate_score_matrix,
    generate_candidates,
    print_top_score_probabilities,
    print_top_candidates,
    print_form_features,
    print_goal_ensemble,
    predict_single_match,
    save_prediction,
)


## Match Input

In [ ]:
team_1 = "Brazil"
team_2 = "Scotland"
stage = "Group Stage"
match_date = "2026-06-24"


## Load Data And Validate Fixture

Fixture validation reads the existing cleaned and future fixture CSVs, normalizes team names, allows either team order, and rejects unresolved placeholders.

In [ ]:
fixtures = load_fixtures()
elo = load_elo()
fifa = load_fifa()
results = load_results(RESULTS_PATH)

fixture = validate_fixture(team_1, team_2, stage, match_date, fixtures, elo, fifa)
fixture


## Build Version 2.5 Features

The form features are computed from historical results strictly before `match_date`.

In [ ]:
feature_row = build_feature_row(fixture, elo, fifa, results)
print_form_features(fixture, feature_row)
feature_row


## Outcome Probabilities And Expected Goals

In [ ]:
outcome_model, goals_team_1_model, goals_team_2_model = load_prediction_models()
ensemble_config = load_goal_ensemble_config()
score_selection_config = load_score_selection_config()
outcome_probabilities = predict_outcome_probabilities(outcome_model, feature_row)
xg_breakdown = predict_ensemble_expected_goals(
    goals_team_1_model,
    goals_team_2_model,
    feature_row,
    ensemble_config,
    goal_scale=score_selection_config["goal_scale"],
)
print_goal_ensemble(xg_breakdown, fixture.team_1, fixture.team_2)

outcome_probabilities, xg_breakdown


## Score Matrix And Selection

Default prediction selects the highest calibrated Poisson score probability. Expected-points ranking is still available for comparison.

In [ ]:
score_matrix = generate_score_matrix(xg_breakdown.calibrated_xg_team_1, xg_breakdown.calibrated_xg_team_2)
candidates = generate_candidates(score_matrix, outcome_probabilities)
print_top_score_probabilities(candidates, limit=5)
print_top_candidates(candidates, limit=5)
candidates.head(10)


## Final Prediction

In [ ]:
prediction = predict_single_match(team_1, team_2, stage, match_date)
save_prediction(prediction)
prediction.__dict__
